In [14]:
import pygame
from classes.logic import Card as card
from classes.logic import Player as player
from classes.logic import Game as game
from classes.logic import Deck as deck
from functions import *
from assets import *
import random
import copy

In [15]:
players = ["AI těžké", "AI těžké", "AI těžké", "AI těžké", "AI těžké"]

In [16]:
g = game.Game(players)

In [17]:
g.season = random.sample(g.seasondeck, 1)[0]
g.seasondeck.remove(g.season)
for p in g.players:
    p.season_buff = g.s_ref[g.season]['akce']
g.hands = []
for i in range(len(g.players)):
    new_hand = deck.PlayerDeck(i, g.game_deck, g.reference)
    for j in new_hand.cardids:
        g.game_deck.remove(j)
    g.hands.append(new_hand)
print(g.round)

1


In [18]:
for p in range(len(g.players)):
                person = g.players[p]
                g.current_player = p
                g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
                d = g.hands[g.current_deck]
                person.start_turn(d, g)
                info = person.choose(d, g)
                id = info[0]
                if d.cards[id].isplayable(person) and info[1] == 'play':
                    karta = d.cards[id]
                    if karta.card_type == "monster" and karta.cards>0:
                            deck.sample_cards(g, person, karta.cards)
                    msg = d.play_card(id, person, g)
                    if msg == 'fialova':
                        person.play_purple(karta)
                    if msg == 'biom':
                        if len(info)>1:
                            person.play_biom_e(karta, info[2])
                else: 
                    d.store_card(id, person, g)
                playing = True
                while playing == True:
                    playing, i = person.want(g)
                    if i is not None:
                        karta = person.stored.cards[i]
                        if karta.card_type == "monster" and karta.cards>0:
                                deck.sample_cards(g, person, karta.cards)
                        msg = person.stored.play_card(i, person, g)
                        if msg == 'fialova':
                            person.play_purple(karta)
                        if msg == 'biom':
                            person.play_biom_e(karta)
                person.had_played = False
g.turn += 1
g.firstplayerdeck = (g.firstplayerdeck+len(g.hands)+(-1)**g.round)%len(g.players)
g.current_deck = (g.current_player+g.firstplayerdeck)%len(g.players)
print(g.turn)


2


In [19]:
print(len(g.game_deck))

166


In [20]:
r =player.Root(g, g.players[0])
r.create_children()
for sim in range(50):
    passable_game = copy.deepcopy(r)
    print(len(passable_game.game_deck))
    node = r
    while node.children != []:
        node = g.players[0].choose_child(node.children, sim, passable_game.current_player)
        node.playout(passable_game)
        node.create_children(passable_game)
    node.end_game(passable_game)
    print(passable_game.results)
    while node.parent is not None:
        node.backpropagate(passable_game.results)
        node = node.parent
    

142
[0.0, 0.3, 0.0, 0.3, 1.0]
142
[0.0, 0.3, 0.0, 0.3, 1.0]
142


IndexError: list index out of range

In [ ]:
r.children

In [ ]:
def create_children(self):
        if self.round == 5:
            return
        if self.biom_card is not None:
            for color in self.colors:
                new_child = Child(self)
                self.biom_card.actually_play(new_child.state.players[new_child.state.current_player], color)
                new_child.state.biom_card = None
                self.children.append(new_child)
        else:
            for c in range(len(self.players[self.current_player].stored.cards)):
                if self.players[self.current_player].stored.cards[c].isplayable(self.players[self.current_player]):
                    new_child = Child(self)
                    played_card = self.players[self.current_player].stored.cards[c]
                    msg = new_child.state.players[self.current_player].stored.play_card(c, new_child.state.players[self.current_player],new_child.state)
                    if played_card.card_type == "monster" and played_card.cards>0:
                        for i in range(played_card.cards):
                            deck.make_card(new_child.state, new_child.state.players[self.current_player], new_child.state.game_deck[0])
                            new_child.state.game_deck.pop(0)
                    if msg == "biom":
                        new_child.state.biom_card = played_card
                    elif msg == "purple":
                        choice = self.choose_purple(played_card)
                        played_card.actually_play(new_child.state.players[self.current_player], choice)
                    self.children.append(new_child)
            if self.players[self.current_player].had_played == False:
                for c in range(len(self.hands[self.current_deck].cards)):
                    if self.hands[self.current_deck].cards[c].isplayable(self.players[self.current_player]):
                        new_child = Child(self)
                        played_card = self.hands[self.current_deck].cards[c]
                        msg = new_child.state.hands[self.current_deck].play_card(c, new_child.state.players[new_child.state.current_player],new_child.state)
                        if played_card.card_type == "monster" and played_card.cards>0:
                            for i in range(played_card.cards):
                                deck.make_card(new_child.state, new_child.state.players[self.current_player], new_child.state.game_deck[0])
                                new_child.state.game_deck.pop(0)
                        if msg == "biom":
                            new_child.state.msg = "biom"
                        elif msg == "purple":
                            choice = self.choose_purple(played_card)
                            played_card.actually_play(new_child.state.players[self.current_player], choice)
                        self.children.append(new_child)
                    new_child = Child(self)
                    new_child.state.hands[self.current_deck].store_card(c, new_child.state.players[new_child.state.current_player],new_child.state)
                    self.children.append(new_child)
            else:
                new_child = Child(self)
                new_child.state.players[new_child.state.current_player].had_played = False
                self.progress(new_child.state)
                self.children.append(new_child)

In [ ]:
if self.round == 5:
            return
        if self.biom_card is not None:
            for color in self.colors:
                new_child = Child(self, ['biom',self.biom_card, color])
                self.children.append(new_child)
        else:
            for c in range(len(self.players[self.current_player].stored.cards)):
                if self.players[self.current_player].stored.cards[c].isplayable(self.players[self.current_player]):
                    new_child = Child(self, ['play_stored', c])
                    self.children.append(new_child)
            if self.players[self.current_player].had_played == False:
                for c in range(len(self.hands[self.current_deck].cards)):
                    if self.hands[self.current_deck].cards[c].isplayable(self.players[self.current_player]):
                        new_child = Child(self, ['play_hand', c])
                        self.children.append(new_child)
                    new_child = Child(self, ['store_hand', c])
                    self.children.append(new_child)
            else:
                new_child = Child(self, ['end_turn', None])
                self.children.append(new_child)

In [ ]:
r.children

In [ ]:
g.players[0].choose_child(r.children, 0)

In [ ]:
dice = random.sample([0,0,0,0,1,2], 1)[0] + random.sample([0,0,0,0,1,2], 1)[0]
goals = []
for play in range(len(g.players)):
        vals = g.players[play].end_season(dice,  g.s_ref[g.season]['akce'])
        goals.append(vals[5])
if goals.count(max(goals))==1:
        index = goals.index(max(goals))
        g.players[index].seasons_won +=1
for person in g.players:
            person.monsters.cards.extend(person.played.cards)
            person.played.cards = []
            person.occupied ={"modra":[], "cerna":[], "hneda":[], "zelena":[], "zlata":[], "fialova":[]}
            for biom in person.bioms:
                person.bioms[biom][0] += person.bioms[biom][1]
                person.bioms[biom][1] = 0
g.round += 1
g.turn = 1

In [ ]:
g.end_game()
print(g.results)